# 1NC9: K19T rescues the SF162 cost of acquiring S56R and T57P

Matched-background pairs of 1NC9 genotypes identical at every site except S56R and T57P, plotted as that background's SF162 affinity with both sites germline against its affinity with both mutated. Points below the diagonal are backgrounds where acquiring the pair is deleterious. Panels split on K19T state; colour gives how many of N52D, S53R and G54R are mutated. Floor-censored measurements are jittered into a band below the floor to distinguish density of overlapping points.

## 1. Setup

In [ ]:
import os
import colorsys
import itertools
import re
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import to_rgb, to_hex
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

%matplotlib inline

## 2. Paths and constants

In [ ]:
# ---- paths -------------------------------------------------------------
data_dir = "../data"
fig_dir = "../figures"
table_dir = "../tables"
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)
PNG_DPI = 300

plt.rcParams["svg.fonttype"] = "path"   # illustrator drops live text
plt.rcParams["pdf.fonttype"] = 42

# display name to file stem; the hyphens and the dot do not appear in any filename
NAME_TO_STEM = {"1NC9": "1NC9", "CH235.12": "CH235_12", "N49P7": "N49P7",
                "NIH45-46": "NIH4546", "VRC-PG20": "VRCPG20", "VRC01": "VRC01",
                "VRC07": "VRC07", "VRC13": "VRC13", "VRC16": "VRC16"}


def save_figure(fig, stem):
    """Write png, pdf and svg, then check the svg has no live text."""
    path = f"{fig_dir}/{stem}"
    fig.savefig(f"{path}.png", dpi=PNG_DPI, bbox_inches="tight")
    fig.savefig(f"{path}.pdf", bbox_inches="tight")
    fig.savefig(f"{path}.svg", bbox_inches="tight")
    assert open(f"{path}.svg").read().count("<text") == 0, "live text in svg, check svg.fonttype"


# ---- constants (inlined from the workflow's config.py) -----------------
ANTIBODIES = ["1NC9", "CH235.12", "VRC01", "VRC13", "VRC16",
              "VRC07", "N49P7", "NIH45-46", "VRC-PG20"]
ANTIGENS = ("CH505TF", "SF162")

GERMLINE_TOKEN = "G"        # site_seq character meaning "germline at this site"
VALUE_COL_RAW = "nlog10_Kd_raw"
FLOOR_CENSOR_VALUE = 6.0    # assay floor; values at exactly this are censored

GERMLINE = GERMLINE_TOKEN
VALUE_COL = VALUE_COL_RAW

## 3. Data loaders

In [ ]:
def load_raw_binding_data():
    """Pooled genotype table assembled from the per-antibody kds files.

    One row per antibody, antigen and genotype, replicate-averaged, with the
    G/M token string in site_seq. Reconstructs what the old pooled
    final_cleaned_all_antibody_binding_data.csv provided.
    """
    frames = []
    for ab in NAME_TO_STEM:
        d = pd.read_csv(f"{data_dir}/{NAME_TO_STEM[ab]}_kds.csv")
        d["antibody"] = ab
        d["site_seq"] = d[[f"site_{i}" for i in range(10)]].agg("".join, axis=1)
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df = (df.groupby(["antibody", "antigen", "site_seq", "id"], as_index=False)["nlog10_Kd"]
            .mean()
            .rename(columns={"nlog10_Kd": "nlog10_Kd_raw"}))
    print(f"Loaded raw binding data: {df.shape} from {len(NAME_TO_STEM)} kds files")
    return df


def find_metadata_file(antibody):
    """Path to one antibody's site metadata."""
    return f"{data_dir}/{NAME_TO_STEM[antibody]}_meta.csv"


def kabat_to_site_index(antibody):
    # {'L45P': 3, ...} -- maps a mutation label to its position in site_seq.
    meta = pd.read_csv(find_metadata_file(antibody))
    meta.columns = meta.columns.str.strip()
    mapping = {}
    for _, row in meta.iterrows():
        site_index = int(str(row["site"]).replace("site_", ""))
        mapping[f"{row['germline_aa']}{row['kabat_position']}{row['mature_aa']}"] = site_index
    return mapping

## 4. Figure code

In [ ]:
matplotlib.use("Agg")


plt.rcParams.update({"font.size": 10.5})


CDR2_OTHER = ["N52D", "S53R", "G54R"]
COUNT_COLORS = {0: "#4363d8", 1: "#3cb44b", 2: "#f58231", 3: "#e6194b"}
FLOOR = FLOOR_CENSOR_VALUE
JITTER_LO, JITTER_HI = 5.5, FLOOR


def jitter_floor(vals, rng):
    vals = vals.copy()
    is_floor = np.isclose(vals, FLOOR)
    vals[is_floor] = rng.uniform(JITTER_LO, JITTER_HI, is_floor.sum())
    return vals


def make_matched_pairs(variant_df, antibody, antigen, k2i, focal_idx, hold_idx=None, rng=None):
    """Matched-background pairs for a single focal site (germline vs mutant),
    optionally requiring a partner site (hold_idx) to be germline throughout."""
    d = variant_df[(variant_df.antibody == antibody) & (variant_df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
    if hold_idx is not None:
        d = d[d["site_seq"].str[hold_idx] == GERMLINE_TOKEN]
    d["_focal"] = d["site_seq"].str[focal_idx] != GERMLINE_TOKEN

    def bgkey(seq):
        return seq[:focal_idx] + seq[focal_idx + 1:]

    d["_bg"] = d["site_seq"].apply(bgkey)
    d["_K19T"] = d["site_seq"].str[k2i["K19T"]] != GERMLINE_TOKEN
    d["_n_cdr2"] = sum((d["site_seq"].str[k2i[m]] != GERMLINE_TOKEN).astype(int) for m in CDR2_OTHER)
    germ = d[~d._focal].groupby("_bg").agg(germ_val=(VALUE_COL_RAW, "mean"), K19T=("_K19T", "first"), n_cdr2=("_n_cdr2", "first"))
    mat = d[d._focal].groupby("_bg").agg(mat_val=(VALUE_COL_RAW, "mean"))
    merged = germ.join(mat, how="inner").reset_index()
    merged["delta"] = merged["mat_val"] - merged["germ_val"]
    if rng is not None:
        merged["germ_plot"] = jitter_floor(merged["germ_val"].values, rng)
        merged["mat_plot"] = jitter_floor(merged["mat_val"].values, rng)
    return merged


def make_joint_pairs(variant_df, antibody, antigen, k2i, i56, i57, rng=None):
    """Matched-background pairs for the JOINT S56R+T57P transition (both germline -> both mutant)."""
    d = variant_df[(variant_df.antibody == antibody) & (variant_df.antigen == antigen)].dropna(subset=[VALUE_COL_RAW]).copy()
    d["_s56"] = d["site_seq"].str[i56] != GERMLINE_TOKEN
    d["_t57"] = d["site_seq"].str[i57] != GERMLINE_TOKEN

    def bgkey(seq):
        return seq[:min(i56, i57)] + seq[min(i56, i57) + 1:max(i56, i57)] + seq[max(i56, i57) + 1:]

    d["_bg"] = d["site_seq"].apply(bgkey)
    d["_K19T"] = d["site_seq"].str[k2i["K19T"]] != GERMLINE_TOKEN
    d["_n_cdr2"] = sum((d["site_seq"].str[k2i[m]] != GERMLINE_TOKEN).astype(int) for m in CDR2_OTHER)
    germ = d[(~d._s56) & (~d._t57)].groupby("_bg").agg(germ_val=(VALUE_COL_RAW, "mean"), K19T=("_K19T", "first"), n_cdr2=("_n_cdr2", "first"))
    mat = d[(d._s56) & (d._t57)].groupby("_bg").agg(mat_val=(VALUE_COL_RAW, "mean"))
    merged = germ.join(mat, how="inner").reset_index()
    merged["delta"] = merged["mat_val"] - merged["germ_val"]
    if rng is not None:
        merged["germ_plot"] = jitter_floor(merged["germ_val"].values, rng)
        merged["mat_plot"] = jitter_floor(merged["mat_val"].values, rng)
    return merged


def discrete_scatter(ax, m, xcol="germ_plot", ycol="mat_plot"):
    for cnt in [0, 1, 2, 3]:
        dd = m[m.n_cdr2 == cnt]
        if len(dd) == 0:
            continue
        ax.scatter(dd[xcol], dd[ycol], color=COUNT_COLORS[cnt], s=32, alpha=0.85, edgecolor="black",
                   linewidth=0.3, zorder=3, label=f"{cnt} of 3 (n={len(dd)})")


def style_scatter(ax, lo, hi, xlabel, title):
    ax.plot([lo, hi], [lo, hi], "--", color="#999999", linewidth=1.2, zorder=1)
    ax.axhline(FLOOR, color="red", linestyle=":", linewidth=0.8, alpha=0.6, zorder=1)
    ax.axvline(FLOOR, color="red", linestyle=":", linewidth=0.8, alpha=0.6, zorder=1)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(xlabel); ax.set_title(title, fontsize=11.5)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.figure.canvas.draw()
    xt = [t for t in ax.get_xticks() if t >= FLOOR - 1e-9 and t <= hi]
    yt = [t for t in ax.get_yticks() if t >= FLOOR - 1e-9 and t <= hi]
    ax.set_xticks(xt); ax.set_yticks(yt)


def stacked_hist_by_count(ax, m, bins):
    for cnt in [0, 1, 2, 3]:
        sub = m[m.n_cdr2 == cnt]["delta"]
        if len(sub) == 0:
            continue
        ax.hist(sub, bins=bins, color=COUNT_COLORS[cnt], alpha=0.6, label=f"{cnt} of 3 (n={len(sub)})")
    ax.axvline(0, color="black", linewidth=1.2, linestyle="--")


def kde_by_count(ax, m, xgrid, min_n=5):
    for cnt in [0, 1, 2, 3]:
        sub = m[m.n_cdr2 == cnt]["delta"].values
        if len(sub) < min_n:
            continue
        kde = gaussian_kde(sub, bw_method="scott")
        density = kde(xgrid)
        ax.plot(xgrid, density, color=COUNT_COLORS[cnt], linewidth=2, zorder=3)
        ax.fill_between(xgrid, density, color=COUNT_COLORS[cnt], alpha=0.25, zorder=2)
        ax.plot(sub, np.full_like(sub, -0.02 * ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else -0.01), "|",
                color=COUNT_COLORS[cnt], markersize=8, markeredgewidth=1.3, alpha=0.8, zorder=4,
                clip_on=False, transform=ax.transData)
        ax.plot([], [], color=COUNT_COLORS[cnt], linewidth=6, alpha=0.5, label=f"{cnt} of 3 (n={len(sub)})")
    ax.axvline(0, color="black", linewidth=1.2, linestyle="--", zorder=5)

## 5. Make the figure

In [ ]:
rng = np.random.default_rng(42)          # fixed seed: floor-censored points are jittered
df = load_raw_binding_data()
k2i = kabat_to_site_index("1NC9")
i56, i57 = k2i["S56R"], k2i["T57P"]

# Matched background pairs for the JOINT S56R+T57P mutation, on SF162.
m = make_joint_pairs(df, "1NC9", "SF162", k2i, i56, i57, rng)
print(f"{len(m)} matched background pairs")

lo = JITTER_LO - 0.1
hi = max(m.germ_plot.max(), m.mat_plot.max()) + 0.1

fig, axes = plt.subplots(1, 2, figsize=(13, 6), dpi=175, sharex=True, sharey=True)
for ax, k19_state, tag in zip(axes, [False, True], ["K19T germline", "K19T mutated"]):
    dd = m[m.K19T == k19_state]
    discrete_scatter(ax, dd)
    style_scatter(ax, lo, hi, "SF162: S56R+T57P GERMLINE", f"SF162, {tag} (n={len(dd)})")
axes[0].set_ylabel("SF162: S56R+T57P MATURE")
axes[1].legend(fontsize=8.5, loc="upper left", bbox_to_anchor=(1.02, 1.0),
               frameon=False, title="# of {N52D,S53R,G54R}")
fig.suptitle("1NC9: K19T rescues the SF162 penalty of acquiring S56R+T57P",
             fontsize=12.5, y=1.03)
plt.tight_layout()
save_figure(fig, "1NC9_K19T_rescue_SF162_scatter")
plt.show()